# COMP 3610: Big Data Analytics - Assignment 4
## MLflow Experiment Tracking, FastAPI Model Serving, and Docker Containerization

This notebook demonstrates a complete machine learning workflow including:
1. **Part 1**: Experiment tracking with MLflow
2. **Part 2**: REST API development with FastAPI
3. **Part 3**: Containerization with Docker and Docker Compose

---

# Part 1: Experiment Tracking with MLflow (25 marks)

## Task 1.1: MLflow Setup & Experiment Logging (10 marks)
Setting up MLflow to track model experiments and log hyperparameters, metrics, artifacts, and tags for reproducibility and model comparison.

### Step 1: Install Required Libraries
Install MLflow and ensure scikit-learn version is compatible with the pre-trained models from Assignment 2. These libraries are essential for experiment tracking, model evaluation, and scikit-learn model serialization.

In [37]:
pip install mlflow scikit-learn==1.6.1

^C
Note: you may need to restart the kernel to use updated packages.


### Step 2: Configure MLflow Tracking Server
Initialize MLflow by pointing to the local tracking server (http://localhost:5000). Create an experiment named "taxi-tip-prediction" to organize all runs for this assignment. This provides a centralized location to store and compare model experiments.

In [2]:
import mlflow
import mlflow.sklearn
import mlflow.pytorch
import joblib
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from mlflow.tracking import MlflowClient

# Point to your local MLflow tracking server
mlflow.set_tracking_uri("http://localhost:5000")

# Create the experiment (or reuse if it already exists)
mlflow.set_experiment("taxi-tip-prediction")

print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment set to: taxi-tip-prediction")

MLflow tracking URI: http://localhost:5000
Experiment set to: taxi-tip-prediction


### Step 3: Load Pre-trained Models and Data
Load the three pre-trained models from Assignment 2 (Random Forest baseline, tuned Random Forest, and preprocessor). Load the yellow taxi dataset for January 2023 from the NYC TLC data source, perform feature engineering including temporal features, geographic zone mapping, and derived features (trip speed, fare per mile, etc.). All features must align with the preprocessor transformations to ensure model compatibility.

In [3]:
# 1. IMPORTS
import joblib
import pandas as pd
import numpy as np

# 2. LOAD MODELS + PREPROCESSOR
rf_reg = joblib.load("models/rf_regressor.joblib")                 # baseline
rf_regressor_tuned = joblib.load("models/rf_regressor_tuned.joblib")  # tuned
preprocessor = joblib.load("models/preprocessor.joblib")

print("Models and preprocessor loaded successfully!")

# 3. LOAD DATA
df = pd.read_parquet(
    "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-01.parquet"
)

df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"])
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"])

# 4. FEATURE ENGINEERING
df["pickup_hour"] = df["tpep_pickup_datetime"].dt.hour
df["pickup_day_of_week"] = df["tpep_pickup_datetime"].dt.weekday
df["is_weekend"] = (df["pickup_day_of_week"] >= 5).astype(int)

df["trip_duration_minutes"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60

# remove invalid rows
df = df[df["trip_duration_minutes"] > 0]

df["log_trip_distance"] = np.log1p(df["trip_distance"])
df["trip_speed_mph"] = df["trip_distance"] / (df["trip_duration_minutes"] / 60)

df["fare_per_mile"] = df["fare_amount"] / df["trip_distance"].replace(0, np.nan)
df["fare_per_minute"] = df["fare_amount"] / df["trip_duration_minutes"].replace(0, np.nan)

# 5. BOROUGH MAPPING
zone_df = pd.read_csv(
    "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"
)

zone_map = dict(zip(zone_df["LocationID"], zone_df["Borough"]))

df["pickup_borough"] = df["PULocationID"].map(zone_map)
df["dropoff_borough"] = df["DOLocationID"].map(zone_map)

df["pickup_borough"] = df["pickup_borough"].fillna("Unknown")
df["dropoff_borough"] = df["dropoff_borough"].fillna("Unknown")

# 6. FEATURES
FEATURES = [
    "passenger_count",
    "trip_distance",
    "fare_amount",
    "total_amount",
    "pickup_hour",
    "pickup_day_of_week",
    "is_weekend",
    "trip_duration_minutes",
    "trip_speed_mph",
    "log_trip_distance",
    "fare_per_mile",
    "fare_per_minute",
    "pickup_borough",
    "dropoff_borough"
]

TARGET_REG = "tip_amount"

df = df.dropna(subset=FEATURES + [TARGET_REG])

X = df[FEATURES]
y_reg = df[TARGET_REG]

# 7. TRANSFORM
X_proc = preprocessor.transform(X)

print("\nExpected features:", rf_reg.n_features_in_)
print("Generated features:", X_proc.shape[1])

assert X_proc.shape[1] == rf_reg.n_features_in_, "Feature mismatch!"

print("Feature alignment successful!")

# 8. PREDICTIONS
y_pred_base = rf_reg.predict(X_proc)
y_pred_tuned = rf_regressor_tuned.predict(X_proc)

# 9. SAMPLE OUTPUT
print("\n--- SAMPLE BASELINE PREDICTIONS ---")
print(y_pred_base[:10])

print("\n--- SAMPLE TUNED PREDICTIONS ---")
print(y_pred_tuned[:10])

Models and preprocessor loaded successfully!

Expected features: 28
Generated features: 28
Feature alignment successful!

--- SAMPLE BASELINE PREDICTIONS ---
[ 0.      4.     13.1515  1.8878  3.28    9.99    3.42   10.5735  5.68
  0.    ]

--- SAMPLE TUNED PREDICTIONS ---
[1.06742369e-04 3.99949673e+00 1.29207960e+01 2.89200314e+00
 3.28000000e+00 9.84509511e+00 3.42000000e+00 1.06986499e+01
 5.68000000e+00 0.00000000e+00]


### Step 4: Train-Test Split
Split the preprocessed data into training (80%) and testing (20%) sets using a fixed random state for reproducibility. This ensures all models are evaluated on the same test set, allowing for fair comparison of their performance metrics.

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_proc, y_reg, test_size=0.2, random_state=42
)

print(f"Train size: {X_train.shape[0]}")
print(f"Test size:  {X_test.shape[0]}")

Train size: 2365268
Test size:  591318


### Step 5: Import Regression Metrics
Import the necessary evaluation metrics (MAE, RMSE, R²) from scikit-learn. These metrics will be logged to MLflow for tracking model performance and are essential for comparing the baseline and tuned Random Forest models.

In [5]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

### Step 6: Define Metrics Logging Function
Create a helper function `log_regression_metrics()` that computes MAE, RMSE, and R² for a given model's predictions. This function centralizes metric calculation and logging to MLflow, ensuring consistent tracking across all model runs. This function satisfies Task 1.1 requirement to log metrics for each run.

In [6]:
def log_regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2 = r2_score(y_true, y_pred)

    mlflow.log_metric("mae", mae)
    mlflow.log_metric("rmse", rmse)
    mlflow.log_metric("r2", r2)

    return mae, rmse, r2

### Step 7: Log Baseline Random Forest Model
Train and evaluate the baseline Random Forest model on the test set. Log all hyperparameters (n_estimators, max_depth, min_samples_split, min_samples_leaf, random_state), compute and log metrics (MAE, RMSE, R²), add tags for model type and version, and log the trained model artifact to MLflow. This fulfills Task 1.1's requirement to log at least 2 models with complete parameter, metric, and artifact tracking.

In [7]:
import mlflow
import mlflow.sklearn

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("taxi-tip-prediction")

with mlflow.start_run(run_name="rf-regressor-baseline"):

    params = {
        "n_estimators": rf_reg.n_estimators,
        "max_depth": rf_reg.max_depth,
        "min_samples_split": rf_reg.min_samples_split,
        "min_samples_leaf": rf_reg.min_samples_leaf,
        "random_state": rf_reg.random_state,
    }
    mlflow.log_params(params)

    preds = rf_reg.predict(X_test)
    mae, rmse, r2 = log_regression_metrics(y_test, preds)

    mlflow.set_tag("model_type", "RandomForest")
    mlflow.set_tag("model_version", "baseline")
    mlflow.set_tag("dataset", "yellow_tripdata_2023_01")

    mlflow.sklearn.log_model(rf_reg, name="model")

    print(f"Logged RF Baseline → MAE={mae:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}")

2026/04/17 16:07:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged RF Baseline → MAE=0.1265, RMSE=0.6846, R2=0.9676
🏃 View run rf-regressor-baseline at: http://localhost:5000/#/experiments/1/runs/6ab785ac5b0b41e9a01d5e9dbe09cedb
🧪 View experiment at: http://localhost:5000/#/experiments/1


### Step 8: Log Tuned Random Forest Model
Train and evaluate the tuned Random Forest model (with improved hyperparameters from hyperparameter tuning in Assignment 2) on the same test set. Log all hyperparameters, metrics (MAE, RMSE, R²), tags, and the model artifact to MLflow. This second logged model allows comparison between baseline and tuned approaches, demonstrating the impact of hyperparameter optimization on model performance.

In [8]:
with mlflow.start_run(run_name="rf-regressor-tuned"):

    # Log the tuned hyperparameters explicitly
    params = {
        "n_estimators": rf_regressor_tuned.n_estimators,
        "max_depth": rf_regressor_tuned.max_depth,
        "min_samples_split": rf_regressor_tuned.min_samples_split,
        "min_samples_leaf": rf_regressor_tuned.min_samples_leaf,
        "random_state": rf_regressor_tuned.random_state,
    }
    mlflow.log_params(params)

    # Evaluate on the same test set
    preds = rf_regressor_tuned.predict(X_test)
    mae, rmse, r2 = log_regression_metrics(y_test, preds)

    # Tags
    mlflow.set_tag("model_type", "RandomForest")
    mlflow.set_tag("model_version", "tuned")
    mlflow.set_tag("dataset", "yellow_tripdata_2023_01")

    # Log the model artifact
    mlflow.sklearn.log_model(rf_regressor_tuned, name= "model")

    print(f"Logged RF Tuned → MAE={mae:.4f}, RMSE={rmse:.4f}, R2={r2:.4f}")

2026/04/17 16:19:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged RF Tuned → MAE=0.1591, RMSE=0.8001, R2=0.9558
🏃 View run rf-regressor-tuned at: http://localhost:5000/#/experiments/1/runs/a6d208324d304873b19fb9e8510a4a6a
🧪 View experiment at: http://localhost:5000/#/experiments/1


### Step 9: Retrieve and Compare All Logged Runs
Query the MLflow experiment to retrieve all logged runs and display them in a side-by-side comparison table. Sort runs by MAE (lowest first) to identify the best-performing model. This step demonstrates MLflow's comparison capabilities and fulfills Task 1.2's requirement to show a side-by-side comparison of all logged runs.

In [36]:
experiment = mlflow.get_experiment_by_name("taxi-tip-prediction")

runs_df = mlflow.search_runs(
    experiment_ids=[experiment.experiment_id],
    order_by=["metrics.mae ASC"]
)

comparison = runs_df[[
    "tags.mlflow.runName",
    "tags.model_version",
    "metrics.mae",
    "metrics.rmse",
    "metrics.r2",
    "run_id",
]].rename(columns={
    "tags.mlflow.runName": "run_name",
    "tags.model_version": "version",
    "metrics.mae": "MAE",
    "metrics.rmse": "RMSE",
    "metrics.r2": "R2",
})

print(comparison.to_string(index=False))

             run_name  version      MAE     RMSE       R2                           run_id
rf-regressor-baseline baseline 0.126547 0.684581 0.967642 6ab785ac5b0b41e9a01d5e9dbe09cedb
   rf-regressor-tuned    tuned 0.159115 0.800056 0.955805 a6d208324d304873b19fb9e8510a4a6a


### MLflow Runs List
![MLflow Runs](screenshots/mlflow_runs.png)

### Step 10: Register Best Model in MLflow Registry
Identify the best-performing model from the comparison (lowest MAE). Register this model in the MLflow Model Registry with a descriptive name ("taxi-tip-regressor") and a comprehensive version description including model type, performance metrics, and dataset information. Transition the model to the "Production" stage for deployment readiness. This fulfills Task 1.2's requirement to register the best model with descriptive metadata.

In [ ]:
from mlflow.tracking import MlflowClient

best_run = runs_df.iloc[0]  # lowest MAE is first
best_run_id = best_run["run_id"]
best_mae   = best_run["metrics.mae"]
best_rmse  = best_run["metrics.rmse"]
best_r2    = best_run["metrics.r2"]
best_ver   = best_run["tags.model_version"]

print(f"Best run: {best_run['tags.mlflow.runName']}")
print(f"  MAE={best_mae:.4f}, RMSE={best_rmse:.4f}, R2={best_r2:.4f}")

# Register
registered = mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/model",
    name="taxi-tip-regressor"
)

# Add description
client = MlflowClient()
client.update_model_version(
    name="taxi-tip-regressor",
    version=registered.version,
    description=(
        f"Best performing model from taxi-tip-prediction experiment. "
        f"Version: {best_ver}. "
        f"MAE={best_mae:.4f}, RMSE={best_rmse:.4f}, R2={best_r2:.4f}. "
        f"Dataset: yellow_tripdata_2023_01. random_state=42."
    )
)

# Transition to Production
client.transition_model_version_stage(
    name="taxi-tip-regressor",
    version=registered.version,
    stage="Production"
)

print(f"\nRegistered as version {registered.version} → Production")

Best run: rf-regressor-baseline
  MAE=0.1265, RMSE=0.6846, R2=0.9676


Registered model 'taxi-tip-regressor' already exists. Creating a new version of this model...
2026/04/17 16:22:32 WARNING mlflow.tracking._model_registry.fluent: Run with id 6ab785ac5b0b41e9a01d5e9dbe09cedb has no artifacts at artifact path 'model', registering model based on models:/m-f23e92c82f0c4612806b2ce8056e1399 instead
2026/04/17 16:22:33 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: taxi-tip-regressor, version 2



Registered as version 2 → Production


Created version '2' of model 'taxi-tip-regressor'.
C:\Users\Taariq\AppData\Local\Temp\ipykernel_15844\480927872.py:33: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


### MLflow Comparison View
![MLflow Comparison](screenshots/mlflow_comparison.png)

### Step 11: Load Production Model from Registry
Load the registered model from the MLflow Model Registry using the "Production" stage alias. This demonstrates the ability to load models from the registry rather than from local files, enabling easy model versioning and deployment. This step is essential for Task 1.2's requirement to demonstrate loading a model from the registry.

In [ ]:
production_model = mlflow.sklearn.load_model("models:/taxi-tip-regressor/Production")
print("Production model loaded successfully!")

Production model loaded successfully!


### Step 12: Demonstration - Sample Prediction with Production Model
Make a sample prediction using the production model loaded from the registry. Compare the model's predicted tip amount against the actual tip amount in the test set. This demonstrates successful model loading and inference capability, fulfilling Task 1.2's requirement to demonstrate model loading and prediction from the registry.

In [ ]:
non_zero_idx = (y_test > 0)
sample = X_test[non_zero_idx][:1]
actual_tip = y_test[non_zero_idx].iloc[0]

predicted_tip = production_model.predict(sample)[0]

print(f"Predicted tip : ${predicted_tip:.2f}")
print(f"Actual tip    : ${actual_tip:.2f}")
print(f"Difference    : ${abs(predicted_tip - actual_tip):.2f}")

Predicted tip : $14.00
Actual tip    : $14.00
Difference    : $0.00


---

# Part 2: Model Serving with FastAPI (35 marks)

Building a production-ready REST API that serves predictions from the registered MLflow model with input validation, error handling, and comprehensive testing.

## Task 2.1: API Design & Implementation (15 marks)

### Step 13: Start FastAPI Server in Background
Start the FastAPI application server using uvicorn on port 8000. The server runs in a background process and exposes the `/predict` endpoint for single predictions and additional endpoints for batch predictions, health checks, and model information. The API loads the model at startup and reuses it across requests for efficiency (Task 2.1 requirement). This setup allows testing the API from subsequent notebook cells.

In [ ]:
import subprocess
import time

api_process = subprocess.Popen(
    ["uvicorn", "app:app", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

time.sleep(3)
print("API running at http://localhost:8000")
print("Swagger docs at http://localhost:8000/docs")

API running at http://localhost:8000
Swagger docs at http://localhost:8000/docs


## Task 2.2: Enhanced API Features (10 marks)

### Step 14: Test Health Check Endpoint
Test the `/health` endpoint (GET) which returns the API status and model loaded status. This is a basic health check endpoint that confirms the API is running and the model is properly loaded. It supports monitoring and orchestration platforms that need to verify service availability (Task 2.2 requirement).

In [ ]:
import requests

response = requests.get("http://localhost:8000/health")
print(response.status_code)
print(response.json())

200
{'status': 'healthy', 'model_loaded': True, 'model_version': '1.0.0', 'uptime_seconds': 8271.0}


### Step 15: Test Single Prediction Endpoint
Test the `/predict` endpoint (POST) with a valid JSON request containing a complete trip record. The request includes all required input fields validated via Pydantic: passenger_count, trip_distance, fare_amount, total_amount, pickup_hour, pickup_day_of_week, is_weekend, trip_duration_minutes, trip_speed_mph, pickup_borough, and dropoff_borough. The response includes the predicted tip amount (rounded to 2 decimals), model_version, and a unique prediction_id. This fulfills Task 2.1's requirement for well-structured request/response schemas with Pydantic validation.

In [ ]:
sample = {
    "passenger_count": 1,
    "trip_distance": 3.5,
    "fare_amount": 14.50,
    "total_amount": 18.80,
    "pickup_hour": 14,
    "pickup_day_of_week": 2,
    "is_weekend": 0,
    "trip_duration_minutes": 12.5,
    "trip_speed_mph": 16.8,
    "pickup_borough": "Manhattan",
    "dropoff_borough": "Brooklyn",
}

response = requests.post("http://localhost:8000/predict", json=sample)
print(response.status_code)
print(response.json())

### Step 16: Test Batch Prediction Endpoint
Test the `/predict/batch` endpoint (POST) with multiple trip records (up to 100). The batch endpoint accepts a JSON object containing a "records" array of trip objects. This endpoint returns a batch response with the count of predictions, processing time in milliseconds, and an array of individual predictions. Batch processing improves efficiency for multiple requests and fulfills Task 2.2's requirement for batch prediction capability.

In [ ]:
batch_payload = {
    "records": [
        {
            "passenger_count": 1,
            "trip_distance": 1.2,
            "fare_amount": 6.50,
            "total_amount": 9.00,
            "pickup_hour": 8,
            "pickup_day_of_week": 0,
            "is_weekend": 0,
            "trip_duration_minutes": 7.0,
            "trip_speed_mph": 10.3,
            "pickup_borough": "Manhattan",
            "dropoff_borough": "Manhattan",
        },
        {
            "passenger_count": 2,
            "trip_distance": 5.7,
            "fare_amount": 19.50,
            "total_amount": 24.00,
            "pickup_hour": 18,
            "pickup_day_of_week": 4,
            "is_weekend": 0,
            "trip_duration_minutes": 25.0,
            "trip_speed_mph": 13.7,
            "pickup_borough": "Queens",
            "dropoff_borough": "Manhattan",
        },
        {
            "passenger_count": 1,
            "trip_distance": 12.3,
            "fare_amount": 38.00,
            "total_amount": 45.50,
            "pickup_hour": 23,
            "pickup_day_of_week": 5,
            "is_weekend": 1,
            "trip_duration_minutes": 40.0,
            "trip_speed_mph": 18.5,
            "pickup_borough": "Manhattan",
            "dropoff_borough": "Brooklyn",
        },
    ]
}

response = requests.post("http://localhost:8000/predict/batch", json=batch_payload)
print(response.status_code)
data = response.json()
print(f"Count: {data['count']}")
print(f"Processing time: {data['processing_time_ms']}ms")
for p in data["predictions"]:
    print(f"  Tip: ${p['predicted_tip_amount']:.2f}  |  ID: {p['prediction_id']}")

200
Count: 3
Processing time: 721.97ms
  Tip: $1.00  |  ID: f66c5f92-5b11-42ff-a0c5-ec05637a1f39
  Tip: $0.24  |  ID: 439e1c9b-c1e7-4685-9f1c-3b29d5310e30
  Tip: $0.64  |  ID: dbb8a3e6-da07-4b54-9b58-848950a1425a


### Step 17: Test Input Validation and Error Handling
Test the API's input validation by sending an invalid request where trip_distance is negative. The API should reject this request with HTTP 422 status code and return detailed validation error messages. Pydantic models enforce constraints such as trip_distance > 0, and pickup_hour in range 0-23. This demonstrates Task 2.1's requirement for input validation and Task 2.2's requirement for error handling with structured error responses.

In [ ]:
sample = {
    "passenger_count": 1,
    "trip_distance": 3.5,
    "fare_amount": 14.50,
    "total_amount": 18.80,
    "pickup_hour": 14,
    "pickup_day_of_week": 2,
    "is_weekend": 0,
    "trip_duration_minutes": 12.5,
    "trip_speed_mph": 16.8,
    "pickup_borough": "Manhattan",
    "dropoff_borough": "Brooklyn",
}

bad_trip = {**sample, "trip_distance": -1.0}

response = requests.post("http://localhost:8000/predict", json=bad_trip)
print(f"Status: {response.status_code}")  # expect 422
print(response.json())

Status: 422
{'detail': [{'type': 'greater_than', 'loc': ['body', 'trip_distance'], 'msg': 'Input should be greater than 0', 'input': -1.0, 'ctx': {'gt': 0.0}}]}


### Step 18: Test Model Info Endpoint
Test the `/model/info` endpoint (GET) which returns comprehensive metadata about the currently loaded model. The response includes model name, version, feature names, and training metrics (MAE, RMSE, R²). This endpoint provides transparency about the model in production and helps users understand what model is making predictions. This fulfills Task 2.2's requirement for a model info endpoint.

In [ ]:
response = requests.get("http://localhost:8000/model/info")
print(response.status_code)
print(response.json())

200
{'model_name': 'taxi-tip-regressor', 'model_version': '1.0.0', 'features': ['passenger_count', 'trip_distance', 'fare_amount', 'total_amount', 'pickup_hour', 'pickup_day_of_week', 'is_weekend', 'trip_duration_minutes', 'trip_speed_mph', 'log_trip_distance', 'fare_per_mile', 'fare_per_minute', 'pickup_borough', 'dropoff_borough'], 'metrics': {'mae': 0.1265, 'rmse': 0.6846, 'r2': 0.9676}, 'trained_on': 'NYC Yellow Taxi Trip Records — January 2023'}


## Task 2.3: API Testing (10 marks)

### Step 19: Run Automated Test Suite
Execute the automated test suite defined in test_app.py using pytest. The test suite includes at least 5 test cases covering:
- Successful single prediction with valid input
- Successful batch prediction
- Invalid input rejection (e.g., negative trip_distance, out-of-range pickup_hour)
- Health check endpoint returns correct status
- Edge cases (zero distance trip or extreme fare values)

All tests must pass, demonstrating comprehensive API functionality, proper input validation, and error handling. This fulfills Task 2.3's requirement for automated testing with pytest and FastAPI TestClient.

In [ ]:
!pip install pytest httpx
!python -m pytest test_app.py -v

============================= test session starts =============================
platform win32 -- Python 3.10.11, pytest-9.0.3, pluggy-1.6.0 -- c:\Users\Taariq\Downloads\a4test\.venv\Scripts\python.exe
cachedir: .pytest_cache
rootdir: c:\Users\Taariq\Downloads\a4test
plugins: anyio-4.13.0
collecting ... collected 20 items

test_app.py::test_root PASSED                                            [  5%]
test_app.py::test_health_check PASSED                                    [ 10%]
test_app.py::test_model_info PASSED                                      [ 15%]
test_app.py::test_predict_valid_input PASSED                             [ 20%]
test_app.py::test_predict_response_is_rounded PASSED                     [ 25%]
test_app.py::test_predict_each_call_gets_unique_id PASSED                [ 30%]
test_app.py::test_predict_batch_valid PASSED                             [ 35%]
test_app.py::test_predict_batch_single_record PASSED                     [ 40%]
test_app.py::test_predict_missing_r

### Swagger UI
![Swagger UI](screenshots/swagger_ui.png)

---

# Part 3: Containerization with Docker (20 marks)

Containerizing the FastAPI application using Docker and orchestrating services with Docker Compose for production-ready deployment.

## Task 3.1: Dockerfile & Image Building (10 marks)

### Step 20: Display Docker Image Information
Display information about the built Docker image including its name, tag, and size. This verifies that the Dockerfile successfully builds a working image. The image should be named "my-ml-api" and based on python:3.11-slim for minimal size. Task 3.1 requires reporting the image size as evidence of successful build.

In [ ]:
result = subprocess.run(
    "docker images my-ml-api",
    capture_output=True, text=True, shell=True
)
print(result.stdout)

IMAGE              ID             DISK USAGE   CONTENT SIZE   EXTRA
my-ml-api:latest   20a54ae4337c       3.56GB          713MB        



### Docker Container Running
![Docker Container](screenshots/docker_container.png)

In [ ]:
import requests

# Request 1
r1 = requests.get("http://localhost:8000/health")
print("Health:", r1.json())

# Request 2
r2 = requests.get("http://localhost:8000/model/info")
print("Model info:", r2.json())

# Request 3
sample = {
    "passenger_count": 1, "trip_distance": 3.5, "fare_amount": 14.50,
    "total_amount": 18.80, "pickup_hour": 14, "pickup_day_of_week": 2,
    "is_weekend": 0, "trip_duration_minutes": 12.5, "trip_speed_mph": 16.8,
    "pickup_borough": "Manhattan", "dropoff_borough": "Brooklyn"
}
r3 = requests.post("http://localhost:8000/predict", json=sample)
print("Prediction:", r3.json())

Health: {'status': 'healthy', 'model_loaded': True, 'model_version': '1.0.0', 'uptime_seconds': 160.1}
Model info: {'model_name': 'taxi-tip-regressor', 'model_version': '1.0.0', 'features': ['passenger_count', 'trip_distance', 'fare_amount', 'total_amount', 'pickup_hour', 'pickup_day_of_week', 'is_weekend', 'trip_duration_minutes', 'trip_speed_mph', 'log_trip_distance', 'fare_per_mile', 'fare_per_minute', 'pickup_borough', 'dropoff_borough'], 'metrics': {'mae': 0.1265, 'rmse': 0.6846, 'r2': 0.9676}, 'trained_on': 'NYC Yellow Taxi Trip Records — January 2023'}
Prediction: {'prediction_id': 'c0e48a49-8bf5-4873-a9aa-e7ff969ba93f', 'predicted_tip_amount': 2.55, 'model_version': '1.0.0'}


## Task 3.2: Docker Compose & Deployment Demo (10 marks)

### Step 21: Create docker-compose.yml File
Generate and write a docker-compose.yml file that orchestrates the complete deployment. The compose file defines:
- **api service**: Builds from the local Dockerfile, maps port 8000:8000, sets environment variables for model and preprocessor paths, includes health checks, and depends on mlflow service
- **mlflow service** (bonus +5 marks): Runs an official MLflow tracking server image on port 5000 with persistent volume storage, enabling full experiment tracking in containerized environments

This setup enables seamless deployment and communication between services through Docker's internal networking.

In [ ]:
compose_content = '''services:

  api:
    build: .
    ports:
      - "8000:8000"
    environment:
      - MODEL_PATH=/app/models/rf_regressor.joblib
      - PREPROCESSOR_PATH=/app/models/preprocessor.joblib
    depends_on:
      - mlflow
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8000/health"]
      interval: 30s
      timeout: 10s
      retries: 3

  mlflow:
    image: ghcr.io/mlflow/mlflow:v2.12.1
    ports:
      - "5000:5000"
    command: mlflow server --host 0.0.0.0 --port 5000
    volumes:
      - mlflow-data:/mlflow

volumes:
  mlflow-data:
'''

with open("docker-compose.yml", "w") as f:
    f.write(compose_content)

print("docker-compose.yml created!")

docker-compose.yml created!


### Step 22: Test Containerized API Directly
Make multiple prediction requests (at least 3) to the containerized FastAPI application running in Docker. Test:
1. Health check endpoint to verify service is running
2. Single prediction endpoint with a valid trip record
3. Batch prediction endpoint with multiple trip records

Each request should return successful responses with correct JSON structure, demonstrating that the containerized API works identically to the non-containerized version. This fulfills Task 3.2's requirement to make at least 3 prediction requests to the containerized API.

In [ ]:
import requests
import time

time.sleep(3)  # give services a moment to fully start

# Request 1 — health check
r1 = requests.get("http://localhost:8000/health")
print("Request 1 — Health check")
print(r1.json())
print()

# Request 2 — single prediction
sample = {
    "passenger_count": 1, "trip_distance": 3.5, "fare_amount": 14.50,
    "total_amount": 18.80, "pickup_hour": 14, "pickup_day_of_week": 2,
    "is_weekend": 0, "trip_duration_minutes": 12.5, "trip_speed_mph": 16.8,
    "pickup_borough": "Manhattan", "dropoff_borough": "Brooklyn"
}
r2 = requests.post("http://localhost:8000/predict", json=sample)
print("Request 2 — Single prediction")
print(r2.json())
print()

# Request 3 — batch prediction
batch = {"records": [sample] * 3}
r3 = requests.post("http://localhost:8000/predict/batch", json=batch)
print("Request 3 — Batch prediction")
data = r3.json()
print(f"Count: {data['count']}")
for p in data["predictions"]:
    print(f"  Tip: ${p['predicted_tip_amount']:.2f}  |  ID: {p['prediction_id']}")

Request 1 — Health check
{'status': 'healthy', 'model_loaded': True, 'model_version': '1.0.0', 'uptime_seconds': 69.0}

Request 2 — Single prediction
{'prediction_id': 'a5e0023d-59b9-4ecd-b481-8aa29ca8f65f', 'predicted_tip_amount': 2.55, 'model_version': '1.0.0'}

Request 3 — Batch prediction
Count: 3
  Tip: $2.55  |  ID: 23175b97-4b07-4b9f-a158-dda369430c45
  Tip: $2.55  |  ID: d96903b6-0027-4147-a04f-1a473c2df02b
  Tip: $2.55  |  ID: c00c37c9-2be7-4e0e-b232-194278bb8576


In [ ]:
result = subprocess.run(
    "docker compose ps",
    capture_output=True, text=True, shell=True
)
print(result.stdout)

NAME              IMAGE                           COMMAND                  SERVICE   CREATED         STATUS                     PORTS
a4test-api-1      a4test-api                      "uvicorn app:app --hâ€¦"   api       2 minutes ago   Up 2 minutes (unhealthy)   0.0.0.0:8000->8000/tcp, [::]:8000->8000/tcp
a4test-mlflow-1   ghcr.io/mlflow/mlflow:v2.12.1   "mlflow server --hosâ€¦"   mlflow    2 minutes ago   Up 2 minutes               0.0.0.0:5000->5000/tcp, [::]:5000->5000/tcp



### Step 23: Verify Docker Compose Services Status
Display the status of all running services managed by docker-compose. The output should show:
- **api** service: Running on port 8000, healthy status from health checks
- **mlflow** service: Running on port 5000 with persistent volume for experiment tracking

This verification demonstrates that the complete multi-container application is functioning correctly and all services can communicate properly. Task 3.2 requires documentation of the container status and configuration.

### Docker Compose Services
![Docker Compose](screenshots/docker_compose_ps.png)

## AI Tools Used

**Claude (Anthropic)** was used throughout this assignment for the following purposes:
- Generating boilerplate code for MLflow experiment tracking, FastAPI endpoints, 
  Dockerfile, and Docker Compose configuration
- Debugging errors encountered during MLflow model registration and API startup
- Structuring the project layout and requirements.txt

All generated code was reviewed, tested, and adapted to fit the specific models, features, 
and dataset used in this assignment. The overall design decisions, model selection rationale, 
and final implementation were my own.